In [76]:
from IPython.display import display

import sys
import os
import requests
import re

import pandas as pd

sys.path.append(os.path.abspath("./"))
import helpers as _

In [30]:
# http://api.steampowered.com/ISteamApps/GetAppList/v0002/?key=STEAMKEY&format=json

df = pd.read_json('Steam/data/games.json')
df['name'] = df['name'].str.strip()

df = df[
    ~df['name'].eq("")
    & ~df['name'].str.startswith('test', na=False) 
    & ~df['name'].str.contains('(?i)demo| test ', na=False) 
]
df

,appid,name
38,2240760,Fantasy Grounds - Pathfinder RPG - Pathfinder ...
40,2240780,307 Racing
41,2240790,Sucker for Love: Date to Die For
43,2240810,Fantasy Grounds - Curse of Ra'khan
44,2240820,Fantasy Grounds - Terrible Beauty
...,...,...
223978,2908120,One Boss One Fight
223979,2774800,FallNation Lost Stories
223981,2562100,Deadlocked
223982,3309620,Lavender Dream


In [93]:
BASE_DIR = 'Steam/data/'

def prepare_headers(content_type='application/json'):
    return {
        "Content-Type": content_type
    }

def call_api_steam_get(method, params=dict(), content_type="application/json"):
    API_BASE_URI = "http://store.steampowered.com/api/"

    response = requests.get(API_BASE_URI + method, params=params, headers=prepare_headers(content_type))
                
    print(f'Response : {response.status_code}')
    
    return response

def call_api_game_details(app_id):
    app_id = str(app_id)
    response = call_api_steam_get('appdetails', {'appids':app_id})

    if response.status_code == 200:
        json = response.json()

        if app_id in json:
            details = json[str(app_id)]

            if details['success']:
                data = details['data']
                
                #print(list(data.keys()))

                platforms = data['platforms']
                is_free = 'is_free' in data and data['is_free']
                is_fullgame = 'fullgame' not in data

                languages = re.sub('<[^<]+?>', '', data['supported_languages'])\
                    .replace('*', '')\
                    .replace('languages with full audio support', '')\
                    .replace('Langues avec support audio complet', '')\
                    .split(',')
                languages = [l.strip() for l in languages]
                

                if platforms['windows'] and not is_free and not is_fullgame:
                    
                    # 'currency': 'EUR', 'initial': 2450, 'final': 1960, 'discount_percent': 20, 'initial_formatted': '24,50€', 'final_formatted': '19,60€'
                    price = data['price_overview']

                    # [{'id': 2, 'description': 'Single-player'}]
                    categories = data['categories']

                    # [{'id': '23', 'description': 'Indie'}]
                    genres = data['genres']

                    columns = ['type', 'steam_appid', 'developers', 'release_date']
    

def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [94]:
for ids in chunk_list(df['appid'].values, 20):
    for id in ids:
        print(f'Query : {id}')
    
        call_api_game_details(id)
                          
    break
    sleep(10)

print('End')

Query : 2240760
Response : 200
Language: ['English']
Query : 2240780
Response : 200
Language: ['Anglais']
Query : 2240790
Response : 200
Language: ['English']
Query : 2240810
Response : 200
Language: ['English']
Query : 2240820
Response : 200
Language: ['English']
Query : 2240890
Response : 200
Language: ['English', 'Turkish']
Query : 2240900
Response : 200
Language: ['AnglaisLangues avec support audio complet']
Query : 2240910
Response : 200
Language: ['English']
Query : 2240940
Response : 200
Language: ['English']
Query : 2240950
Response : 200
Language: ['English', 'German']
Query : 2240980
Response : 200
Language: ['English']
Query : 2241000
Response : 200
Language: ['English', 'French', 'Italian', 'German', 'Czech', 'Dutch', 'Hungarian', 'Polish', 'Portuguese - Brazil', 'Russian', 'Simplified Chinese', 'Spanish - Latin America', 'Turkish']
Query : 2241020
Response : 200
Language: ['English', 'German']
Query : 2241030
Response : 200
Language: ['English', 'Simplified Chinese']
Query